In [ ]:
!pip install tensorflow openpyxl

In [ ]:
# STEP 1: Imports
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# STEP 3: Load the dataset from Google Drive
file_path = "/content/drive/MyDrive/ML/Dummy data set/Per_Trail_EEG-3.xlsx"
df = pd.read_excel(file_path)

In [ ]:
# STEP 4: Prepare features and target
features = [
    'AF7_Alpha', 'AF7_Beta', 'AF7_Theta',
    'AF8_Alpha', 'AF8_Beta', 'AF8_Theta',
    'Theta/Alpha_AF7', 'Alpha/Beta_AF7',
    'Theta/Alpha_AF8', 'Alpha/Beta_AF8'
]
X = df[features]
y = df['Label_Class']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)

In [ ]:
# STEP 5: Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# STEP 6: Build the neural network model
model = Sequential([
    Dense(32, input_dim=X_train_scaled.shape[1], activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# STEP 7: Train model
history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=16, validation_split=0.1, verbose=1)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 220ms/step - accuracy: 0.3672 - loss: 0.7425 - val_accuracy: 0.6000 - val_loss: 0.6846
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5156 - loss: 0.7003 - val_accuracy: 0.6000 - val_loss: 0.6726
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4703 - loss: 0.6827 - val_accuracy: 0.6000 - val_loss: 0.6604
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4516 - loss: 0.6838 - val_accuracy: 0.6000 - val_loss: 0.6485
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.5328 - loss: 0.6688 - val_accuracy: 0.6000 - val_loss: 0.6381
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.5969 - loss: 0.6370 - val_accuracy: 0.8000 - val_loss: 0.6279
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.7375 - loss: 0.6267 - val_accuracy: 0.8000 - val_loss: 0.6178
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.7297 - loss: 0.5971 - val_accuracy: 0.8000 - val_loss: 0.6

In [ ]:
# STEP 8: Evaluate and predict
loss, accuracy = model.evaluate(X_test_scaled, y_test)
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

print("Test Accuracy:", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.8333 - loss: 0.4096
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
Test Accuracy: 0.8333333134651184


In [ ]:
# STEP 9: Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.75      1.00      0.86         6
           1       1.00      0.67      0.80         6

    accuracy                           0.83        12
   macro avg       0.88      0.83      0.83        12
weighted avg       0.88      0.83      0.83        12



In [ ]:
import numpy as np
import random
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)


n_splits = 5       # Number of folds
n_repeats = 5      # Number of times to repeat k-fold CV
epochs = 100
batch_size = 16

early_stop = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=0)

# Lists to store metrics
all_accuracies = []
all_precisions = []
all_recalls = []
all_f1s = []

y_train_np = y_train.values


for repeat in range(n_repeats):
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + repeat)
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled, y_train_np), 1):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train_np[train_idx], y_train_np[val_idx]

        # Neural network model
        model = Sequential([
            Dense(64, activation='relu', input_shape=(X_tr.shape[1],)),
            BatchNormalization(),
            Dropout(0.3),
            Dense(32, activation='relu'),
            BatchNormalization(),
            Dropout(0.25),
            Dense(16, activation='relu'),
            Dense(1, activation='sigmoid')
        ])

        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        # Train model
        model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                  epochs=epochs, batch_size=batch_size, callbacks=[early_stop], verbose=0)

        # Predictions
        y_pred = (model.predict(X_val) > 0.5).astype(int).flatten()

        # Metrics
        acc = np.mean(y_pred == y_val)
        prec = precision_score(y_val, y_pred, zero_division=0)
        rec = recall_score(y_val, y_pred, zero_division=0)
        f1 = f1_score(y_val, y_pred, zero_division=0)

        # Append
        all_accuracies.append(acc)
        all_precisions.append(prec)
        all_recalls.append(rec)
        all_f1s.append(f1)


def mean_std_ci(arr):
    arr = np.array(arr)
    mean = np.mean(arr)
    std = np.std(arr)
    sem = stats.sem(arr)
    ci_low, ci_high = stats.norm.interval(0.95, loc=mean, scale=sem)
    return mean, std, ci_low, ci_high

acc_mean, acc_std, acc_ci_low, acc_ci_high = mean_std_ci(all_accuracies)
prec_mean, prec_std, _, _ = mean_std_ci(all_precisions)
rec_mean, rec_std, _, _ = mean_std_ci(all_recalls)
f1_mean, f1_std, _, _ = mean_std_ci(all_f1s)

print("\n==============================")
print(f"Accuracy: {acc_mean:.2f} ± {acc_std:.2f}")
print(f"Precision: {prec_mean:.2f} ± {prec_std:.2f}")
print(f"Recall: {rec_mean:.2f} ± {rec_std:.2f}")
print(f"F1-Score: {f1_mean:.2f} ± {f1_std:.2f}")
print(f"Accuracy (95% CI): {acc_mean:.2f} ({acc_ci_low:.2f} - {acc_ci_high:.2f})")
print("==============================")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step

Accuracy: 0.79 ± 0.12
Precision: 0.90 ± 0.13
Recall: 0.67 ± 0.27
F1-Score: 0.72 ± 0.19
Accuracy (95% CI): 0.79 (0.74 - 0.84)
